In [ ]:


# ============================================================
# MODULE 4 — PANDAS WRANGLING & EDA
# ============================================================
# Using CLEANED datasets from Module 2


import pandas as pd
import numpy as np

# 1. LOADED CLEANED DATASETS

# 2. CONVERT DATE COLUMNS
# ============================================================

customers["SignupDate"] = pd.to_datetime(
    customers["SignupDate"],
    errors="coerce"
)

subscriptions["StartDate"] = pd.to_datetime(
    subscriptions["StartDate"],
    errors="coerce"
)

subscriptions["EndDate"] = pd.to_datetime(
    subscriptions["EndDate"],
    errors="coerce"
)

usage["Month"] = pd.to_datetime(
    usage["Month"],
    errors="coerce"
)


# 3. CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = {
    "subscriptions": ["MRR", "Seats"],
    "usage": ["Logins", "ActiveUsers", "APICalls", "SessionMinutes"],
    "tickets": ["ResolutionHours", "SatisfactionScore"]
}

for column in numeric_columns["subscriptions"]:
    subscriptions[column] = pd.to_numeric(
        subscriptions[column],
        errors="coerce"
    )

for column in numeric_columns["usage"]:
    usage[column] = pd.to_numeric(
        usage[column],
        errors="coerce"
    )

for column in numeric_columns["tickets"]:
    tickets[column] = pd.to_numeric(
        tickets[column],
        errors="coerce"
    )


# CUSTOMERS — NO NUMERIC COLUMNS TO CONVERT
# ============================================================

print("\nCustomers: No numeric columns specified for conversion.")


# 4. CREATE CUSTOMER-LEVEL SUBSCRIPTION SUMMARY
# ============================================================

subscription_summary = (
    subscriptions
    .groupby("CustomerID")
    .agg(
        Total_MRR=("MRR", "sum"),
        Average_MRR=("MRR", "mean"),
        Total_Seats=("Seats", "sum"),
        Subscription_Count=("SubscriptionID", "nunique")
    )
    .reset_index()
)

print("\nSubscription summary:")
print(subscription_summary.head())


# 5. CREATE CUSTOMER-LEVEL USAGE SUMMARY
# ============================================================

usage_summary = (
    usage
    .groupby("CustomerID")
    .agg(
        Total_Logins=("Logins", "sum"),
        Average_ActiveUsers=("ActiveUsers", "mean"),
        Total_APICalls=("APICalls", "sum"),
        Total_SessionMinutes=("SessionMinutes", "sum"),
        Usage_Records=("Month", "count")
    )
    .reset_index()
)

print("\nUsage summary:")
print(usage_summary.head())


# ============================================================
# 6. CREATE CUSTOMER-LEVEL TICKET SUMMARY
# ============================================================

ticket_summary = (
    tickets
    .groupby("CustomerID")
    .agg(
        Ticket_Count=("TicketID", "nunique"),
        Average_ResolutionHours=("ResolutionHours", "mean"),
        Average_Satisfaction=("SatisfactionScore", "mean")
    )
    .reset_index()
)

print("\nTicket summary:")
print(ticket_summary.head())


Customers: No numeric columns specified for conversion.

Subscription summary:
  CustomerID  Total_MRR  Average_MRR  Total_Seats  Subscription_Count
0      C1001     278.60       278.60          6.0                   1
1      C1002     246.76       246.76          4.0                   1
2      C1003     278.60       278.60          6.0                   1
3      C1004      80.36        80.36          9.0                   1
4      C1005     618.76       618.76          4.0                   1

Usage summary:
  CustomerID  Total_Logins  Average_ActiveUsers  Total_APICalls  \
0      C1001           148             4.500000           10276   
1      C1002           217            12.181818           15512   
2      C1003           187             7.250000           19771   
3      C1004            82             4.200000           14012   
4      C1005           133             4.692308           13232   

   Total_SessionMinutes  Usage_Records  
0                2241.4             16  

In [ ]:
# 7. MERGE ALL FOUR TABLES
# ============================================================

# Customers is the master/customer table.
# LEFT JOIN is used because every customer should remain
# in the final customer-level analytical view, even if that
# customer has no subscription, usage, or ticket records.

customer_view = customers.merge(
    subscription_summary,
    on="CustomerID",
    how="left"
)

customer_view = customer_view.merge(
    usage_summary,
    on="CustomerID",
    how="left"
)

customer_view = customer_view.merge(
    ticket_summary,
    on="CustomerID",
    how="left"
)


# Customers with no related activity receive zero
# for count/sum-based metrics.

zero_columns = [
    "Total_MRR",
    "Average_MRR",
    "Total_Seats",
    "Subscription_Count",
    "Total_Logins",
    "Average_ActiveUsers",
    "Total_APICalls",
    "Total_SessionMinutes",
    "Usage_Records",
    "Ticket_Count"
]

for column in zero_columns:
    customer_view[column] = customer_view[column].fillna(0)


print("\n" + "=" * 70)
print("CUSTOMER-LEVEL VIEW")
print("=" * 70)

print("Shape:", customer_view.shape)
print("\nColumns:")
print(customer_view.columns.tolist())

print("\nFirst 5 rows:")
print(customer_view.head())



CUSTOMER-LEVEL VIEW
Shape: (420, 20)

Columns:
['CustomerID', 'CompanyName', 'Industry', 'Country', 'City', 'EmployeeCount', 'SignupDate', 'AcquisitionChannel', 'Total_MRR', 'Average_MRR', 'Total_Seats', 'Subscription_Count', 'Total_Logins', 'Average_ActiveUsers', 'Total_APICalls', 'Total_SessionMinutes', 'Usage_Records', 'Ticket_Count', 'Average_ResolutionHours', 'Average_Satisfaction']

First 5 rows:
  CustomerID               CompanyName   Industry Country        City  \
0      C1001      Ishaan Joshi Systems    Unknown   India   Hyderabad   
1      C1002  Tara Sharma Technologies    Finance      UK      London   
2      C1003          Zoya Das Systems    Finance     UAE       Dubai   
3      C1004           Diya Joshi Labs     Retail      UK  Manchester   
4      C1005        Tara Joshi Systems  Logistics      UK      London   

   EmployeeCount SignupDate AcquisitionChannel  Total_MRR  Average_MRR  \
0           80.0 2023-07-02           Paid Ads     278.60       278.60   
1     

In [ ]:
# 8. GROUPBY — PLAN
# ============================================================

plan_analysis = (
    subscriptions
    .groupby("PlanName")
    .agg(
        Customers=("CustomerID", "nunique"),
        Subscriptions=("SubscriptionID", "nunique"),
        Total_MRR=("MRR", "sum"),
        Average_MRR=("MRR", "mean"),
        Total_Seats=("Seats", "sum"),
        Average_Seats=("Seats", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("GROUPBY — PLAN")
print("=" * 70)
print(plan_analysis)


# 9. GROUPBY — INDUSTRY
# ============================================================

industry_analysis = (
    customer_view
    .groupby("Industry")
    .agg(
        Customers=("CustomerID", "nunique"),
        Total_MRR=("Total_MRR", "sum"),
        Average_MRR=("Total_MRR", "mean"),
        Total_Seats=("Total_Seats", "sum"),
        Total_Logins=("Total_Logins", "sum"),
        Total_Tickets=("Ticket_Count", "sum")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("GROUPBY — INDUSTRY")
print("=" * 70)
print(industry_analysis)


# 10. GROUPBY — COUNTRY
# ============================================================

region_analysis = (
    customer_view
    .groupby("Country")
    .agg(
        Customers=("CustomerID", "nunique"),
        Total_MRR=("Total_MRR", "sum"),
        Average_MRR=("Total_MRR", "mean"),
        Total_Seats=("Total_Seats", "sum"),
        Total_Logins=("Total_Logins", "sum"),
        Total_Tickets=("Ticket_Count", "sum")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("GROUPBY — COUNTRY")
print("=" * 70)
print(region_analysis)


# 11. GROUPBY — ACQUISITION CHANNEL
# ============================================================

channel_analysis = (
    customer_view
    .groupby("AcquisitionChannel")
    .agg(
        Customers=("CustomerID", "nunique"),
        Total_MRR=("Total_MRR", "sum"),
        Average_MRR=("Total_MRR", "mean"),
        Total_Seats=("Total_Seats", "sum"),
        Total_Logins=("Total_Logins", "sum"),
        Total_Tickets=("Ticket_Count", "sum")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("GROUPBY — ACQUISITION CHANNEL")
print("=" * 70)
print(channel_analysis)


GROUPBY — PLAN
     PlanName  Customers  Subscriptions  Total_MRR  Average_MRR  Total_Seats  \
0    Business         85             85   61132.60   719.207059        567.0   
1  Enterprise         69             71  149229.88  2101.829296        454.0   
2      Growth        145            146   42920.32   293.974795       1020.0   
3     Starter        132            135   10082.76    74.687111        965.0   

   Average_Seats  
0       6.670588  
1       6.394366  
2       6.986301  
3       7.148148  

GROUPBY — INDUSTRY
        Industry  Customers  Total_MRR  Average_MRR  Total_Seats  \
0      Education         51   32491.88   637.095686        392.0   
1        Finance         67   43823.76   654.085970        499.0   
2     Healthcare         51   31562.48   618.872157        346.0   
3      Logistics         64   30678.72   479.355000        433.0   
4  Manufacturing         59   44832.48   759.872542        435.0   
5          Media         55   30772.96   559.508364        3

In [ ]:
# 12. PIVOT TABLE — PLAN × BILLING TERM
# ============================================================

pivot_plan_billing = pd.pivot_table(
    subscriptions,
    index="PlanName",
    columns="BillingTerm",
    values="MRR",
    aggfunc="sum",
    fill_value=0
)

print("\n" + "=" * 70)
print("PIVOT — PLAN × BILLING TERM")
print("=" * 70)
print(pivot_plan_billing)


# 13. PIVOT TABLE — INDUSTRY × ACQUISITION CHANNEL
# ============================================================

pivot_industry_channel = pd.pivot_table(
    customer_view,
    index="Industry",
    columns="AcquisitionChannel",
    values="Total_MRR",
    aggfunc="sum",
    fill_value=0
)

print("\n" + "=" * 70)
print("PIVOT — INDUSTRY × ACQUISITION CHANNEL")
print("=" * 70)
print(pivot_industry_channel)


PIVOT — PLAN × BILLING TERM
BillingTerm    Annual   Monthly
PlanName                       
Business     24186.64  36945.96
Enterprise   57081.92  92147.96
Growth       17973.68  24946.64
Starter       3420.20   6662.56

PIVOT — INDUSTRY × ACQUISITION CHANNEL
AcquisitionChannel   Events  Organic Search  Outbound  Paid Ads   Partner  \
Industry                                                                    
Education           1900.68        12162.28   3480.64   6778.68   4017.08   
Finance             4708.20         9694.80   5596.16   7121.84   5439.84   
Healthcare          6578.68         2268.44   2578.28  10138.16   6472.52   
Logistics           2226.04         3740.32   5577.00  12898.36   4167.68   
Manufacturing       1738.84        11653.28   2040.84  10042.24  10101.40   
Media               1978.68         8926.28    359.20   9503.24   2720.28   
Retail              2262.44        17469.88   3426.76   8783.40   3051.36   
Unknown                0.00         1450.64   

In [ ]:
# 14. CALCULATED COLUMN — TENURE
# ============================================================

# Tenure is calculated in months from SignupDate to the
# latest available date in the dataset.

analysis_date = max(
    customers["SignupDate"].max(),
    subscriptions["StartDate"].max(),
    usage["Month"].max()
)

customer_view["Tenure_Months"] = (
    (analysis_date - customer_view["SignupDate"])
    .dt.days / 30.44
).round(2)

print("\nTenure created successfully.")


# 15. CALCULATED COLUMN — REVENUE PER SEAT
# ============================================================

customer_view["Revenue_Per_Seat"] = np.where(
    customer_view["Total_Seats"] > 0,
    customer_view["Total_MRR"] / customer_view["Total_Seats"],
    0
)

print("Revenue_Per_Seat created successfully.")


# 16. CALCULATED COLUMN — TICKETS PER MONTH
# ============================================================

customer_view["Tickets_Per_Month"] = np.where(
    customer_view["Tenure_Months"] > 0,
    customer_view["Ticket_Count"] /
    customer_view["Tenure_Months"],
    0
)

print("Tickets_Per_Month created successfully.")


Tenure created successfully.
Revenue_Per_Seat created successfully.
Tickets_Per_Month created successfully.


In [ ]:
# 17. CALCULATED COLUMN — USAGE TREND
# ============================================================

# Divide usage records into first half and second half
# of the available usage period.

usage_customer = usage.copy()

usage_customer["Usage_Period"] = np.where(
    usage_customer["Month"] <=
    usage_customer["Month"].median(),
    "First Half",
    "Second Half"
)

usage_trend = (
    usage_customer
    .groupby(["CustomerID", "Usage_Period"])
    .agg(
        Total_Logins=("Logins", "sum"),
        Total_ActiveUsers=("ActiveUsers", "sum"),
        Total_APICalls=("APICalls", "sum")
    )
    .reset_index()
)

usage_pivot = usage_trend.pivot(
    index="CustomerID",
    columns="Usage_Period",
    values="Total_Logins"
).fillna(0)

usage_pivot.columns.name = None

if "First Half" not in usage_pivot.columns:
    usage_pivot["First Half"] = 0

if "Second Half" not in usage_pivot.columns:
    usage_pivot["Second Half"] = 0

usage_pivot["Usage_Change"] = (
    usage_pivot["Second Half"] -
    usage_pivot["First Half"]
)

usage_pivot["Usage_Trend"] = np.where(
    usage_pivot["Usage_Change"] > 0,
    "Increasing",
    np.where(
        usage_pivot["Usage_Change"] < 0,
        "Decreasing",
        "Stable"
    )
)

customer_view = customer_view.merge(
    usage_pivot[["Usage_Trend"]],
    on="CustomerID",
    how="left"
)

customer_view["Usage_Trend"] = (
    customer_view["Usage_Trend"]
    .fillna("No Usage Data")
)

print("Usage_Trend created successfully.")

Usage_Trend created successfully.


In [ ]:
# 18. IQR OUTLIER DETECTION
# ============================================================

def detect_iqr_outliers(df, column):
    """
    Detect outliers using the IQR method.

    Q1 = 25th percentile
    Q3 = 75th percentile
    IQR = Q3 - Q1

    Lower Limit = Q1 - 1.5 × IQR
    Upper Limit = Q3 + 1.5 × IQR
    """

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower_limit = q1 - (1.5 * iqr)
    upper_limit = q3 + (1.5 * iqr)

    outlier_mask = (
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    )

    print("\n" + "-" * 60)
    print(f"IQR OUTLIER ANALYSIS — {column}")
    print("-" * 60)

    print("Q1:", round(q1, 2))
    print("Q3:", round(q3, 2))
    print("IQR:", round(iqr, 2))
    print("Lower Limit:", round(lower_limit, 2))
    print("Upper Limit:", round(upper_limit, 2))
    print("Outliers:", outlier_mask.sum())

    return outlier_mask


# ============================================================
# DETECT OUTLIERS IN IMPORTANT BUSINESS METRICS
# ============================================================

customer_view["MRR_Outlier"] = detect_iqr_outliers(
    customer_view,
    "Total_MRR"
)

customer_view["Seats_Outlier"] = detect_iqr_outliers(
    customer_view,
    "Total_Seats"
)

customer_view["Login_Outlier"] = detect_iqr_outliers(
    customer_view,
    "Total_Logins"
)

customer_view["Tickets_Outlier"] = detect_iqr_outliers(
    customer_view,
    "Ticket_Count"
)

print("\nIQR outlier detection completed successfully.")


------------------------------------------------------------
IQR OUTLIER ANALYSIS — Total_MRR
------------------------------------------------------------
Q1: 84.28
Q3: 738.52
IQR: 654.24
Lower Limit: -897.08
Upper Limit: 1719.88
Outliers: 63

------------------------------------------------------------
IQR OUTLIER ANALYSIS — Total_Seats
------------------------------------------------------------
Q1: 5.0
Q3: 9.0
IQR: 4.0
Lower Limit: -1.0
Upper Limit: 15.0
Outliers: 9

------------------------------------------------------------
IQR OUTLIER ANALYSIS — Total_Logins
------------------------------------------------------------
Q1: 33.0
Q3: 159.25
IQR: 126.25
Lower Limit: -156.38
Upper Limit: 348.62
Outliers: 23

------------------------------------------------------------
IQR OUTLIER ANALYSIS — Ticket_Count
------------------------------------------------------------
Q1: 2.0
Q3: 4.0
IQR: 2.0
Lower Limit: -1.0
Upper Limit: 7.0
Outliers: 7

IQR outlier detection completed successfully.


In [ ]:
# 19. CORRELATION MATRIX
# ============================================================

correlation_columns = [
    "Total_MRR",
    "Total_Seats",
    "Total_Logins",
    "Average_ActiveUsers",
    "Total_APICalls",
    "Total_SessionMinutes",
    "Ticket_Count",
    "Average_ResolutionHours",
    "Average_Satisfaction",
    "Tenure_Months",
    "Revenue_Per_Seat",
    "Tickets_Per_Month"
]

correlation_matrix = customer_view[
    correlation_columns
].corr()

print("\n" + "=" * 70)
print("CORRELATION MATRIX")
print("=" * 70)

print(correlation_matrix.round(3))


CORRELATION MATRIX
                         Total_MRR  Total_Seats  Total_Logins  \
Total_MRR                    1.000        0.147         0.035   
Total_Seats                  0.147        1.000        -0.027   
Total_Logins                 0.035       -0.027         1.000   
Average_ActiveUsers          0.068       -0.033         0.610   
Total_APICalls              -0.055       -0.003         0.694   
Total_SessionMinutes        -0.004       -0.026         0.746   
Ticket_Count                 0.052        0.003         0.084   
Average_ResolutionHours      0.009        0.115        -0.066   
Average_Satisfaction        -0.044       -0.015         0.074   
Tenure_Months               -0.031       -0.012         0.635   
Revenue_Per_Seat             0.743       -0.245         0.030   
Tickets_Per_Month            0.063        0.034        -0.225   

                         Average_ActiveUsers  Total_APICalls  \
Total_MRR                              0.068          -0.055   
Total_

In [ ]:
# 20. PRINT MEANINGFUL CORRELATIONS
# ============================================================

print("\n" + "=" * 70)
print("MEANINGFUL CORRELATIONS")
print("=" * 70)

for i in range(len(correlation_matrix.columns)):

    for j in range(i + 1, len(correlation_matrix.columns)):

        variable_1 = correlation_matrix.columns[i]
        variable_2 = correlation_matrix.columns[j]

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= 0.30:

            if correlation >= 0.70:
                strength = "strong positive"

            elif correlation >= 0.50:
                strength = "moderate positive"

            elif correlation >= 0.30:
                strength = "weak positive"

            elif correlation <= -0.70:
                strength = "strong negative"

            elif correlation <= -0.50:
                strength = "moderate negative"

            else:
                strength = "weak negative"

            print(
                f"{variable_1} vs {variable_2}: "
                f"{correlation:.3f} → {strength} relationship"
            )


MEANINGFUL CORRELATIONS
Total_MRR vs Revenue_Per_Seat: 0.743 → strong positive relationship
Total_Logins vs Average_ActiveUsers: 0.610 → moderate positive relationship
Total_Logins vs Total_APICalls: 0.694 → moderate positive relationship
Total_Logins vs Total_SessionMinutes: 0.746 → strong positive relationship
Total_Logins vs Tenure_Months: 0.635 → moderate positive relationship
Total_APICalls vs Total_SessionMinutes: 0.898 → strong positive relationship
Total_APICalls vs Tenure_Months: 0.763 → strong positive relationship
Total_SessionMinutes vs Tenure_Months: 0.830 → strong positive relationship
Total_SessionMinutes vs Tickets_Per_Month: -0.319 → weak negative relationship
Ticket_Count vs Tickets_Per_Month: 0.855 → strong positive relationship
Average_ResolutionHours vs Average_Satisfaction: -0.409 → weak negative relationship
Tenure_Months vs Tickets_Per_Month: -0.389 → weak negative relationship


In [ ]:
# 21. SAVE MODULE 4 OUTPUTS
# ============================================================

customer_view.to_csv(
    "module4_customer_level_view.csv",
    index=False
)

plan_analysis.to_csv(
    "module4_plan_analysis.csv",
    index=False
)

industry_analysis.to_csv(
    "module4_industry_analysis.csv",
    index=False
)

region_analysis.to_csv(
    "module4_region_analysis.csv",
    index=False
)

channel_analysis.to_csv(
    "module4_channel_analysis.csv",
    index=False
)

pivot_plan_billing.to_csv(
    "module4_pivot_plan_billing.csv",
    index=False
)

pivot_industry_channel.to_csv(
    "module4_pivot_industry_channel.csv",
    index=False
)

correlation_matrix.to_csv(
    "module4_correlation_matrix.csv",
    index=False
)


# ============================================================
# MODULE 4 COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("MODULE 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFiles saved:")
print("1. module4_customer_level_view.csv")
print("2. module4_plan_analysis.csv")
print("3. module4_industry_analysis.csv")
print("4. module4_region_analysis.csv")
print("5. module4_channel_analysis.csv")
print("6. module4_pivot_plan_billing.csv")
print("7. module4_pivot_industry_channel.csv")
print("8. module4_correlation_matrix.csv")


MODULE 4 COMPLETED SUCCESSFULLY

Files saved:
1. module4_customer_level_view.csv
2. module4_plan_analysis.csv
3. module4_industry_analysis.csv
4. module4_region_analysis.csv
5. module4_channel_analysis.csv
6. module4_pivot_plan_billing.csv
7. module4_pivot_industry_channel.csv
8. module4_correlation_matrix.csv
